# 06 指标 (`core.metrics`)

In [ ]:
import os, sys
from pathlib import Path

_project = os.path.abspath(os.path.join(os.getcwd(), ".."))
if _project.replace("\\", "/") not in [p.replace("\\", "/") for p in sys.path]:
    sys.path.insert(0, _project)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from hscredit import init_setting
init_setting()


def load_demo_excel():
    roots = [Path.cwd(), Path.cwd() / "examples", Path.cwd().parent]
    for root in roots:
        for name in ("hengshucredit_yyp.xlsx", "hscredit_yyp.xlsx"):
            p = root / name
            if p.is_file():
                return pd.read_excel(p), p
    raise FileNotFoundError(
        "请将 hengshucredit_yyp.xlsx 或 hscredit_yyp.xlsx 放在 examples/ 目录。"
    )


df, DATA_PATH = load_demo_excel()
print("数据文件:", DATA_PATH, "形状:", df.shape)

if "MOB1" in df.columns:
    df = df.copy()
    df["target_demo"] = (pd.to_numeric(df["MOB1"], errors="coerce").fillna(-1) > 3).astype(int)
    target = "target_demo"
elif "FPD" in df.columns:
    df = df.copy()
    t = pd.to_numeric(df["FPD"], errors="coerce")
    df = df.loc[t.notna()].copy()
    df["target_demo"] = (t.loc[df.index] > 0).astype(int)
    target = "target_demo"
else:
    raise ValueError("需要 MOB1 或 FPD 列")

exclude = {"MOB1", "MOB2", "target_demo", "CURRENT_DPD"}
num_cols = [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]
print("目标:", target, "坏率:", f"{df[target].mean():.2%}")


In [ ]:
import numpy as np
from hscredit.core import metrics
from hscredit.core.metrics import composite_binning_quality
from hscredit import OptimalBinning

rng = np.random.default_rng(2)
y_true = df[target].to_numpy()
p = rng.uniform(0, 1, size=len(df))
print('KS:', metrics.ks(y_true, p), 'AUC:', metrics.auc(y_true, p))
feat = num_cols[0]
x = pd.to_numeric(df[feat], errors='coerce').fillna(0)
print('IV:', metrics.iv(y_true, x))
b = OptimalBinning(method='mdlp', max_n_bins=5, verbose=False)
b.fit(df[[feat]], df[target])
bins = b.transform(df[[feat]], metric='indices')[feat].to_numpy()
print('composite_binning_quality:', composite_binning_quality(bins, y_true))
